# Strands + LangSmith: Tracing, PromptHub & Evaluations

This notebook demonstrates a **Strands-native** agent (with tools) using LangSmith for:

1. **OTEL Tracing** — Strands spans exported to LangSmith via a custom exporter
2. **PromptHub** — Push the agent's system prompt to Hub, then pull it back to create the agent
3. **Datasets & Evaluators** — Create a QA dataset and wire up an LLM-judge evaluator
4. **Experiments** — Run the agent against the dataset with `client.evaluate()`
5. **Traced Agent Call** — A standalone demo showing tool use in traces

The agent (`agent.py`) has three tools: `lookup_knowledge_base`, `calculator`, and `web_search`. Model provider is configured in `setup/config.py` via `create_model()` — default is **OpenAI**, swap to **Bedrock** by commenting/uncommenting.

In [1]:
!uv sync

Resolved 172 packages in 1ms
Audited 166 packages in 0.06ms


## 0. Setup — Config & OTEL Tracing

Load environment variables and wire up the custom LangSmith OTEL exporter. This must run before any Strands agent calls so that spans are captured.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Wire up OTEL tracing: Strands spans → LangSmith
from langsmith_exporter import setup_langsmith_telemetry
setup_langsmith_telemetry()

from setup.config import client, DATASET_NAME, PROJECT_NAME

## 1. Push Agent Prompt to LangSmith Hub

Push the agent's system prompt to Hub via `client.push_prompt()`. This is versioned — update the text in `setup/prompts.py` and re-run to push a new commit.

In [3]:
from setup.prompts import push_agent_prompt

push_agent_prompt()

  Pushed prompt: https://smith.langchain.com/prompts/strands-research-assistant/ab30ec42?organizationId=58636190-0252-4526-9dc7-0b09b37b499c


'https://smith.langchain.com/prompts/strands-research-assistant/ab30ec42?organizationId=58636190-0252-4526-9dc7-0b09b37b499c'

## 2. Create Agent (prompt pulled from Hub)

Pull the system prompt from PromptHub and use it to create the Strands agent. This means you can update the prompt in Hub and get a new agent behavior without changing any code.

In [4]:
from setup.prompts import pull_agent_prompt
from agent import create_agent, ask

# Pull the system prompt from LangSmith Hub
system_prompt = pull_agent_prompt()
print(f"Pulled system prompt ({len(system_prompt)} chars):\n{system_prompt[:200]}...")

# Create the agent with the Hub prompt
agent = create_agent(system_prompt)

Pulled system prompt (739 chars):
You are a research assistant that helps answer technical questions about AI/ML tools and platforms. You have access to:

- **lookup_knowledge_base**: Query the internal knowledge base for information ...


## 3. Create Evaluation Dataset

A QA dataset with 6 examples that exercise the agent's tools (knowledge base, calculator, multi-tool). Idempotent — skips if it already exists.

In [5]:
from setup.datasets import create_dataset

create_dataset()

  Created dataset 'Strands Research QA' with 6 examples.


## 4. Create LLM-Judge Evaluator

Creates a **correctness** evaluator on the dataset with an inline prompt (system + human messages). The evaluator runs automatically on new experiment results.

In [6]:
from setup.evaluators import create_llm_judge_evaluator

create_llm_judge_evaluator()

  Created LLM-judge evaluator 'correctness' on dataset.


## 5. Run Evaluation Experiment

Runs the agent on every dataset example via `client.evaluate()`. Each call produces OTEL traces. We include a **`no_pii`** code evaluator that checks the output for leaked PII (emails, phone numbers, SSNs, credit cards).

In [7]:
import re

def run_agent(inputs: dict) -> dict:
    """Target function for client.evaluate() — create agent with Hub prompt, run it."""
    eval_agent = create_agent(system_prompt)
    return {"answer": ask(eval_agent, inputs["question"])}


# PII patterns: email, phone (US), SSN, credit card
PII_PATTERNS = [
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",  # email
    r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",                          # phone
    r"\b\d{3}-\d{2}-\d{4}\b",                                   # SSN
    r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b",            # credit card
]

def no_pii(outputs: dict, reference_outputs: dict) -> dict:
    """Code evaluator: check that the agent output contains no PII."""
    text = outputs.get("answer", "")
    for pattern in PII_PATTERNS:
        if re.search(pattern, text):
            return {"key": "no_pii", "score": False}
    return {"key": "no_pii", "score": True}


results = client.evaluate(
    run_agent,
    data=DATASET_NAME,
    evaluators=[no_pii],
    experiment_prefix="strands-research-qa",
    num_repetitions=1,
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")

View the evaluation results for experiment: 'strands-research-qa-91499821' at:
https://smith.langchain.com/o/58636190-0252-4526-9dc7-0b09b37b499c/datasets/62868c35-5130-4e92-83b7-cf25b576dda1/compare?selectedSessions=23331265-84b6-4783-a720-901880445e82




0it [00:00, ?it/s]


Tool #1: calculator

Tool #1: lookup_knowledge_base
The URL for the Strands Agents SDK repository is [https://github.com/strands-agents/sdk-python](https://github.com/
Tool #2: calculator
strands-agents/sdk-python).The result of \(2^{16}\) is
Tool #1: lookup_knowledge_base
 65,536, and the square root of that result is 256.0.Strands Agents supports multiple model providers, including Amazon Bedrock, OpenAI, and Anthropic. For more detailed information, you can visit their [GitHub page](https://github.com/strands-agents/sdk-python).
Tool #1: lookup_knowledge_base

Tool #1: calculator
LangSmith is a platform designed for building production-grade large language model (LLM) applications. It offers several key features:

1. **Tracing & Observability**: LangSmith provides tools to trace and observe the performance and behavior of LLM applications.
2. **Evaluation & Testing**: It includes capabilities for evaluating and testing language models to ensure they meetWith 3 Bedrock desired perfo

## 6. Traced Agent Demo

Reuse the agent created in step 2. This exercises multiple tools — check the LangSmith project to see the full trace.

In [8]:
# This question exercises multiple tools: knowledge base lookup + calculator
output = ask(agent, "What is Strands Agents and what providers does it support? Also, if I run 5 agents each handling 200 requests/hour, how many requests is that per day?")
print(output)


Tool #1: lookup_knowledge_base

Tool #2: calculator
**Strands Agents** is an open-source SDK designed for building AI agents. It supports a variety of model providers, including Amazon Bedrock, OpenAI, and Anthropic. Strands Agents offers features such as multi-provider support, tool usage, streaming capabilities, and OpenTelemetry tracing [knowledge base].

Regarding the number of requests: If you run 5 agents, each handling 200 requests per hour, the total number of requests per day would be 24,000 [calculator].**Strands Agents** is an open-source SDK designed for building AI agents. It supports a variety of model providers, including Amazon Bedrock, OpenAI, and Anthropic. Strands Agents offers features such as multi-provider support, tool usage, streaming capabilities, and OpenTelemetry tracing [knowledge base].

Regarding the number of requests: If you run 5 agents, each handling 200 requests per hour, the total number of requests per day would be 24,000 [calculator].



## 7. Cleanup (optional)

Delete all LangSmith resources created by this demo: evaluators, dataset, prompts, and project.

In [9]:
from setup.cleanup import cleanup_all

# Uncomment to delete all resources:
# cleanup_all()